# Chest X-ray — FULL run (all 7 models) — CRASH-SAFE

All 7 models, full epochs. Crash safety built in (Cell 3 symlinks results/ to Drive; epoch backups to Drive). Disconnect mid-run -> reconnect, re-run Cell 8, it continues. Run directly for 7-from-start, or after colab_small returns GO.

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone / update code

In [ ]:
import os
REPO='https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    !git clone $REPO /content/Chest_Xray
else:
    !git -C /content/Chest_Xray pull
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — CRASH-SAFETY: point results/ AT Google Drive (BEFORE anything writes)
Every checkpoint, JSON and figure now lands on Drive the INSTANT it is written. A disconnect no longer loses results — reconnect, re-run, and `--resume` skips finished models.

In [ ]:
import os, shutil
DRIVE='/content/drive/MyDrive/cxr_data'
RES=f'{DRIVE}/results'; BACKUP=f'{DRIVE}/backup'
os.makedirs(RES,exist_ok=True); os.makedirs(BACKUP,exist_ok=True)
if os.path.islink('results'): os.remove('results')
elif os.path.isdir('results'): shutil.rmtree('results')
os.symlink(RES,'results')
print('results ->', os.path.realpath('results')); print('backup  ->', BACKUP)

## Cell 4 — unzip data to fast local disk (slow, once per session)

In [ ]:
import os, glob
DRIVE='/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw',exist_ok=True)
for z in glob.glob(f'{DRIVE}/*.zip'):
    print('unzipping', os.path.basename(z), flush=True)
    !unzip -q -o "$z" -d data/raw
print('folders:', os.listdir('data/raw'))

## Cell 5 — bring the manifest

In [ ]:
import os, shutil
DRIVE='/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE}/manifest.csv'):
    shutil.copy(f'{DRIVE}/manifest.csv','data/manifest.csv'); print('copied')
else:
    !python scripts/build_manifest.py

## Cell 6 — install deps + GPU check (GPU must NOT be empty)

In [ ]:
!pip -q install -r requirements.txt
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 7 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df=pd.read_csv('data/manifest.csv')
missing=[p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
print('>>> OK: continue.' if not missing else f'>>> STOP. example: {missing[0]} ; folders: {os.listdir("data/raw")}')

## Cell 8 — pipeline on all 7 models (CRASH-SAFE, LIVE, resumable)
Long (hours). Safe to disconnect — re-run after reconnect; continues where it stopped.

In [ ]:
MODELS='densenet201 efficientnetb0 resnet50 mobilenetv3large xception vit lenet5'
BACKUP='/content/drive/MyDrive/cxr_data/backup'
!python -u scripts/run_pipeline.py --models $MODELS --epochs 100 --batch-size 64 --backup-dir $BACKUP

## (optional) remove a stale 1-epoch smoke checkpoint before a real full run

In [ ]:
# !rm -rf results/lenet5   # uncomment ONLY if results/lenet5 holds the smoke checkpoint

## Cell 9 — final C3 coupling (all 7)

In [ ]:
import json
print(json.dumps(json.load(open('results/c3_coupling.json')),indent=2))

## Cell 10 — final outputs (already on Drive)

In [ ]:
import os
print('FIGURES:', os.listdir('results/figures'))
print('TABLES :', os.listdir('results/tables'))
print('STATS  :', os.path.exists('results/stats_summary.json'))